#### Notebook que muestre el judge de los modelos

In [6]:
#-----importación de librerías
from pathlib import Path
import sys
import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT/ ".env")

from src.generation.groq_llm import GroqLLm
from src.generation.gemini_llm import GeminiLLm
from src.generation.ollama_llm import OllamaLLm
from src.rag.pipeline import RAGPipeline
from src.evaluation.questions import EVAL_QUESTIONS
from src.evaluation.judge import judge_answer
import warnings
warnings.filterwarnings("ignore")

In [7]:
#------el juez gemini
juez = GeminiLLm()

In [8]:
#-------llamada a los modelos implementados
modelos = {
    "Groq": RAGPipeline(llm=GroqLLm()),
    "Gemini": RAGPipeline(llm=GeminiLLm()),
    "Ollama": RAGPipeline(llm=OllamaLLm())
}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
#-----recorremos todos los modelos y realizamos las mismas preguntasrespondibles
preguntas_eval = [q for q in EVAL_QUESTIONS if q["category"]=="answerable"]
filas = []
for nombre, rag in modelos.items():
    print(f"-----Juzgando respuestas de {nombre}----")
    for q in preguntas_eval:
        resultado = rag.answer(q["question"])
        scores = judge_answer(
            juez,
            q["question"],
            resultado["contexto"],
            resultado["respuesta"],
        )
        filas.append({
            "modelo":nombre,
            "id": q["id"],
            "pregunta": q["question"],
            "fidelidad": scores["fidelidad"],
            "relevancia": scores["relevancia"],
            "respuesta": resultado["respuesta"],
        })
     
df_judge = pd.DataFrame(filas)
df_judge


-----Juzgando respuestas de Groq----
-----Juzgando respuestas de Gemini----
-----Juzgando respuestas de Ollama----


,modelo,id,pregunta,fidelidad,relevancia,respuesta
0,Groq,1,¿Qué medicamentos del contexto se utilizan par...,1,1,**Medicamentos que se utilizan para tratar la ...
1,Groq,2,¿Qué medicamentos sirven para tratar la diabetes?,5,5,Los medicamentos mencionados en el contexto qu...
2,Groq,3,¿Qué medicamentos del contexto se utilizan par...,5,5,**Medicamentos que se utilizan para tratar ale...
3,Groq,4,¿Qué medicamentos se usan para la acidez estom...,5,5,Los medicamentos indicados para la acidez esto...
4,Groq,5,¿Qué efectos secundarios tienen los medicament...,5,5,Los efectos secundarios de los medicamentos pa...
5,Groq,6,¿Qué medicamentos del contexto tratan infeccio...,5,5,"Los medicamentos que, según el contexto, se ut..."
6,Gemini,1,¿Qué medicamentos del contexto se utilizan par...,5,5,"Basado en el contexto proporcionado, los medic..."
7,Gemini,2,¿Qué medicamentos sirven para tratar la diabetes?,5,5,"Basado en la información proporcionada, los me..."
8,Gemini,3,¿Qué medicamentos del contexto se utilizan par...,1,1,"Basado en el contexto proporcionado, los medic..."
9,Gemini,4,¿Qué medicamentos se usan para la acidez estom...,5,5,"Basado en la información proporcionada, los me..."


In [10]:
resumen_judge = df_judge.groupby("modelo").agg(
    fidelidad_media=("fidelidad","mean"),
    relevancia_media=("relevancia","mean"),
).round(2)

resumen_judge

,fidelidad_media,relevancia_media
modelo,,
Gemini,3.67,3.67
Groq,4.33,4.33
Ollama,3.00,3.33


#### Conclusiones

Con la finalidad de complementar las métricas de latencia y abstención, se utilizó un LLM como judge para evaluar la calidad de las respuestas generadas por los tres modelos. La evaluación se realizó únicamente sobre las respuestas de la categoría "answerable".

Se evaluaron dos criterios en una escala de 1 a 5:

**Fidelidad:** qué tan respaldada está la respuesta en el contexto

**Relevancia:** qué tan directamente responde el modelo a la pregunta realizada.

- **Groq** obtuvo los mejores resultados, con **4.33/5** en fidelidad y relevancia.
- **Gemini** mostró un desempeño intermedio, con **3.67/5** en ambas métricas.
- **Ollama** obtuvo los resultados más bajos, con **3.00/5** en fidelidad y **3.33/5** en relevancia.

En esta evaluación, **Groq presentó el mejor equilibrio en la calidad de las respuestas generadas**.

**Nota:** El usar a Gemini como juez y modelo introduce una especie de sesgo potencial ya que podría favorecerse a sí mismo. Como una mejora del proyecto sería implementar un cuarto modelo neutral como LLM-judge